<div dir="rtl">

# رافِدَين — تدريب Seed-VC على اللهجة العراقية على Google Colab (نسخة 2)

دفتر Colab واحد متكامل يصل من جلسة GPU فارغة إلى نموذج
`seed-uvit-tat-xlsr-tiny` مضبوط على الكلام العراقي، محفوظاً بشكل **دائم** في
Google Drive و HuggingFace Hub:

| المرحلة | ماذا يحدث |
|---|---|
| 1. البيئة | توصيل Drive، فحص GPU/القرص، تثبيت ffmpeg + حزم بايثون **داخل نواة الجلسة** (PyTorch 2.4.0 cu121) — بلا بيئة معزولة |
| 2. المستودعات | استنساخ Seed-VC + رافِدَين من GitHub، ثم **استيراد وحداتهما مباشرة** (بلا سطر أوامر) |
| 3. البيانات الجاهزة | تنزيل أرشيف **`rafidain_current_sources_ready.tar.gz`** (4.89GB) — بيانات تدريب مقطّعة ومرمّزة Opus 24kHz جاهزة، **بلا إعادة تنزيل من المصادر ولا إعادة معالجة** |
| 4. التدريب | ضبط دقيق **داخل نواة الدفتر نفسها** عبر `Trainer` المستورد (`import train`)، مع مرآة نقاط فورية إلى Drive عند كل حفظ |
| 5. التحقق والتصدير | استدلال حقيقي بالنقطة الناتجة (استماع داخل الدفتر)، ثم حزمة تصدير (أوزان + إعداد + تقرير) |
| 6. الحفظ الدائم | نسخة في **Google Drive** + أرشيف tar.gz + **رفع إلى HuggingFace Hub** (دائم فعلاً) |

> **ما الجديد في النسخة 2 ولماذا؟** النسخة السابقة كانت تُطلق التدريب كعملية خلفية
> مستقلة عن النواة، مع مراقب نسخ دائم وحلقة انتظار لساعات — وهذا النمط يُصنَّف
> «استخداماً غير مسموح» في طبقة Colab المجانية فتُقطع الجلسة فجأة
> (Runtime disconnected). هنا **كل شيء يعمل داخل الخلية نفسها**: التدريب في المقدمة
> بمخرجات حية، والنسخ إلى Drive يحدث لحظياً عند كل حفظ نقطة من داخل عملية التدريب
> نفسها — لا عمليات خفية ولا حلقات إبقاء الجلسة.

> كل خلية **آمنة لإعادة التشغيل**: ما أُنجز يُتخطى (بصمة الأرشيف + فحوص وجود)،
> والتدريب يستأنف من آخر نقطة حفظ. صُمم لطبيعة Colab: جلسات منقطعة، قرص محلي
> مؤقت، وDrive دائم.

**المصدر**: [github.com/Alqudimi/rafidain](https://github.com/Alqudimi/rafidain) ·
النموذج: [Plachta/Seed-VC](https://huggingface.co/Plachta/Seed-VC)‏ (`seed-uvit-tat-xlsr-tiny`)

</div>

<div dir="rtl">

## ١) تجهيز Colab (قبل تشغيل الخلايا)

### أ. اختيار الـ GPU
من القائمة: **Runtime ← Change runtime type ← Hardware accelerator: GPU** ثم Save.

| GPU | كيف تحصل عليه | الذاكرة | حجم الدفعة المقترح | ملاحظات |
|---|---|---|---|---|
| **T4** | المجاني الافتراضي | 16GB | 2 | يعمل كاملاً — 3000 خطوة في نحو 2–4 ساعات |
| **L4** | Colab Pro | 24GB | 4 | أسرع بنحو الضعف من T4 |
| **A100 40GB** | Colab Pro / Pro+ | 40GB | 6 | فائض عن حاجة نموذج 25M باراميتر |

الدفتر يضبط حجم الدفعة تلقائياً حسب الذاكرة المكتشفة (الخلية 3) — لا تحتاج تعديل شيء.

### ب. حدود جلسة Colab — وكيف يتعامل معها هذا الدفتر (بصراحة)
- الجلسة المجانية تعيش **حتى ~12 ساعة**، وتُقطع بعد خمول النواة.
- **فلسفة النسخة 2**: لا محاولة «خداع» البقاء — التدريب يعمل في مقدمة الخلية
  (النواة مشغولة بعمل حقيقي ومخرجاته تظهر أمامك مباشرة)، وكل نقطة حفظ تُنسخ
  **فوراً** إلى Drive أثناء التدريب نفسه. لو انقطعت الجلسة في أي لحظة، خسارتك
  القصوى = الخطوات منذ آخر نقطة حفظ (250 خطوة افتراضياً ≈ دقائق).
- بعد أي انقطاع: **Runtime ← Run all** — البيئة تُبنى، والأرشيف الجاهز يُستخدم
  (بصمته محفوظة فلا يُعاد تنزيله إن وُجد)، والخلية 13 تستعيد آخر نقطة من Drive،
  والتدريب يكمل من عندها.

### ج. Google Drive — طبقة التخزين الدائم
- تحتاج **~2GB** من حصتك (آخر نقطتين ≈ 0.4GB لكل نقطة + النموذج النهائي + السجلات
  + حزمة التصدير). حصة Drive المجانية (15GB) تكفي تماماً.
- بيانات التدريب الثقيلة تبقى على **القرص المحلي السريع** `/content` — عمداً:
  Drive بطيء جداً لآلاف الملفات الصغيرة، والأرشيف الجاهز يُعاد تنزيله بسرعة عند الحاجة.

### د. قرص Colab المحلي
| العنصر | الحجم |
|---|---|
| الأرشيف الجاهز (يُنزَّل ويُفك ثم يُحذف المضغوط) | 4.9GB تنزيلاً ← ≈ 5.2GB بعد الفك |
| بيئة Python + PyTorch | ≈ 5GB |
| نقاط النموذج الأساسية | ≈ 2.2GB |
| نقاط التدريب + الحزمة | ≈ 1.5GB |
| **الإجمالي التقريبي** | **≈ 14GB** |

القرص المحلي في Colab عادة 100GB+ — دخول مريح جداً (النسخة السابقة كانت تحتاج
حتى 42GB لأنها كانت تجلب المصادر الخام وتعالجها؛ الأرشيف الجاهز ألغى ذلك كله).

### هـ. فتح هذا الدفتر
نزّل ملف `rafidain_colab_training.ipynb` وارفعه إلى [colab.research.google.com](https://colab.research.google.com)
(أو ضعه في Drive وافتحه بـ *Open with ← Colab*)، فعّل GPU كما في (أ)، ثم شغّل الخلايا
من الأعلى للأسفل.

</div>

<div dir="rtl">

## ٢) خريطة الدفتر — شغّل بالترتيب من الأعلى

| # | الخلية | الوصف | المدة التقريبية |
|---|---|---|---|
| 1 | توصيل Google Drive | طبقة التخزين الدائم (تخطَّها إن أردت العمل بلا Drive) | ثوانٍ |
| 2 | الإعدادات المركزية | كل المقابض + رابط الأرشيف الجاهز — تُحفظ على Drive | ثوانٍ |
| 3 | فحص البيئة | GPU/قرص/أدوات + تحديد حجم الدفعة تلقائياً | ثوانٍ |
| 4 | تبعيات النظام | ffmpeg + git (apt فقط — لا بيئة معزولة) | أقل من دقيقة |
| 5 | حزم بايثون داخل النواة | تثبيت الإصدارات المطلوبة في نواة الجلسة نفسها — **بلا بيئة منفصلة وبلا عملية خارجية للمنطق** | 2–6 دقائق |
| 6 | المستودعات + الاستيراد | استنساخ Seed-VC + رافِدَين ثم **استيراد** خط البيانات والمدرب والاستدلال | أقل من دقيقة |
| 7 | البيانات الجاهزة | تنزيل أرشيف `rafidain_current_sources_ready.tar.gz` وفكّه — **بلا أي إعادة معالجة** | 5–20 دقيقة (مرة واحدة) |
| 8 | بياناتك الخاصة | (اختياري) مجلد لملفاتك الصوتية يُقطَّع ويُدمج في التدريب | حسبك |
| 9 | إحصاءات البيانات | تقرير الساعات/الملفات لكل مصدر (من فهرس الأرشيف) | ثوانٍ |
| 10 | نقاط النموذج | تنزيل DiT/XLSR/HiFT/CAMPPlus مسبقاً (نحو 2.2GB) — عبر استيراد خط البيانات | 2–10 دقائق |
| 11 | استعادة من Drive | (بعد أي انقطاع) إعادة آخر نقطة تدريب من Drive | ثوانٍ |
| 12 | التدريب (في المقدمة) | `Trainer` المستورد يعمل **داخل نواة الدفتر مباشرة** + مرآة Drive عند كل حفظ | 2–4 ساعات (T4) |
| 13 | التحقق | استدلال حقيقي بـ `inference` المستورد **داخل النواة** + استماع في الدفتر | 1–3 دقائق |
| 14 | التصدير | حزمة أوزان + إعداد + تقرير → ونسخها إلى Drive | ثوانٍ |
| 15 | أرشيف tar.gz | ملف واحد في Drive جاهز للتنزيل | ثوانٍ |
| 16 | HuggingFace Hub | **التخزين السحابي الدائم** — النموذج في حسابك للأبد | 1–5 دقائق |
| 17 | تنظيف المساحة | (اختيارية) حذف الأرشيف المفكوك والبيئة المؤقتة | ثوانٍ |
| 18 | النشر والاستخدام | أوامر جاهزة: استدلال فوري + خدمة رافِدَين على نفس الجلسة | اختيارية |

**قواعد ذهبية**
- كل الخلايا **آمنة لإعادة التشغيل** — لا شيء يُعاد تنزيله أو تنفيذه إن كان منجزاً
  (بصمة الأرشيف + علامات `.done` + فحوص وجود).
- إذا انقطعت الجلسة: افتح الدفتر وشغّل **Run all** — الحزم تُثبَّت، والأرشيف يُفك
  فوراً من نسخة Drive إن وُجدت، والخلية 11 تستعيد نقاط التدريب، والتدريب يكمل من
  آخر نقطة حفظ.
- **لا توجد عمليات خلفية إطلاقاً** في هذا الدفتر — كل خلية تنجز عملها وتنتهي.
  هذا تحديداً هو ما يمنع قطع الجلسة «disallowed usage» في الطبقة المجانية.
- **كل منطق المشروع يعمل داخل نواة الدفتر بالاستيراد المباشر** — التدريب
  (`import train` ← `Trainer`)، والاستدلال (`import inference` ← `main()`)، وخط
  البيانات ورفع HuggingFace. لا عمليات خارجية إطلاقاً لمنطق المشروع؛ الأوامر
  الخارجية الوحيدة هي أدوات النظام نفسها (apt/git/tar/curl) لتركيب الأدوات
  واستنساخ المستودعات وفك الأرشيف — ثم يُستخدم كل ما نُزّل عبر الاستيراد.
- **لا تعتمد على القرص المحلي `/content` للديمومة** — النقاط تُنسخ إلى Drive أثناء
  التدريب لحظياً، والحزمة النهائية تُنسخ وتُؤرشف وتُرفع في الخلايا الأخيرة.

</div>

In [ ]:
# ============================================================ #
#  الخلية ١ — توصيل Google Drive (طبقة التخزين الدائم)         #
#  هنا تُحفظ: الإعدادات + نقاط التدريب + حزم النموذج النهائية   #
#  (يمكن تخطيها — الدفتر يعمل بلا Drive لكن لا شيء ينجو من      #
#   إعادة تعيين الجلسة إلا مرفوعاً إلى HuggingFace لاحقاً)      #
# ============================================================ #
from pathlib import Path
import shutil

DRIVE_MNT = Path("/content/drive")
DRIVE_OK, DRIVE_ROOT = False, None

try:
    from google.colab import drive          # متاح على Colab فقط
    drive.mount(str(DRIVE_MNT))
    DRIVE_OK = (DRIVE_MNT / "MyDrive").is_dir()
except ModuleNotFoundError:
    print("[Drive] لست على Colab — تخطّي توصيل Drive (العمل محلي بالكامل).")
except Exception as e:
    print(f"[Drive] تعذر التوصيل: {e} — تابع بلا Drive أو أعد هذه الخلية.")

if DRIVE_OK:
    DRIVE_ROOT = DRIVE_MNT / "MyDrive" / "rafidain_seedvc"
    for sub in ("runs", "exports", "data"):
        (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
    try:
        du = shutil.disk_usage(DRIVE_ROOT)
        print(f"[Drive] موصول — التخزين الدائم: {DRIVE_ROOT}")
        print(f"[Drive] الحصة: حر {du.free/1e9:.1f}GB من {du.total/1e9:.1f}GB (النسخ الاحتياطية تحتاج ~2GB)")
    except Exception:
        print(f"[Drive] موصول — التخزين الدائم: {DRIVE_ROOT}")
    print("[Drive] ما يُحفظ هنا: nb_config.json + runs/<اسم-التشغيل>/ + exports/<اسم-التشغيل>/ + data/ (الأرشيف الجاهز)")
else:
    print(R"""
[تحذير] العمل بلا Drive: كل ما يبقى على /content يضيع عند انقطاع الجلسة.
         يبقى خيارك الدائم الوحيد حينها الرفع إلى HuggingFace Hub (الخلية 16).
         لتوصيل Drive لاحقاً: شغّل هذه الخلية ثم أعد تشغيل الخلية 2.""")

In [ ]:
# ============================================================ #
#  الخلية ٢ — الإعدادات المركزية والأدوات المساعدة              #
#  كل مقابض التحكم في مكان واحد، وتُحفظ على Drive إن وُجد       #
#  (إعادة تشغيل النواة/الجلسة لا تفقدها — وتُعاد عند الاستئناف) #
# ============================================================ #
import os, sys, json, shutil, time, re
from pathlib import Path

# ---------- 1) حالة Drive من الخلية 1 (آمنة بعد إعادة تشغيل النواة) ----------
DRIVE_OK   = globals().get("DRIVE_OK", False)
DRIVE_ROOT = globals().get("DRIVE_ROOT", None)

# ---------- 2) جذر العمل: القرص المحلي السريع دائماً (Drive بطيء للبيانات!) ----------
ROOT = Path(os.environ.get("IRAQI_DATA_ROOT", "").strip() or "/content/iraqi_seedvc")
ROOT.mkdir(parents=True, exist_ok=True)

# الإعدادات تُحفظ على Drive إن كان موصولاً — فتنجو من إعادة تعيين الجلسة
CFG_DIR  = DRIVE_ROOT if (DRIVE_OK and DRIVE_ROOT) else ROOT
CFG_PATH = CFG_DIR / "nb_config.json"

# ---------- 3) المقابض الافتراضية ----------
DEFAULTS = dict(
    # البيانات الجاهزة (الأرشيف المُجهَّز مسبقاً — لا تنزيل من المصادر ولا إعادة معالجة)
    READY_ARCHIVE_URL = (
        "https://drive.usercontent.google.com/download"
        "?id=1fFCx7wvabl34Z6RtYrGEYvy9zrx5U8Du&export=download&authuser=0"
        "&confirm=t&uuid=cf4cf0ad-5338-4949-af9d-c61e27fa1bf1"
        "&at=AMrWOn35Bq4LFlvhoCZfaklOunHk%3A1794664871926"
    ),   # رابط مباشر لأرشيف rafidain_current_sources_ready.tar.gz (4.89GB)
    READY_ARCHIVE_SIZE = 4890474991,  # الحجم الدقيق بالبايت — للتحقق من اكتمال التنزيل
    READY_DIR_NAME = "rafidain_current_sources_ready",  # اسم المجلد الجذري داخل الأرشيف
    # التدريب
    RUN_NAME       = "iraqi_v1",  # اسم التشغيل (مجلد النقاط والحزمة)
    STEPS          = 3000,   # عدد خطوات الضبط الدقيق (500 لمتحدث واحد، 1500-4000 للهجة عموماً)
    BATCH          = "auto", # حجم الدفعة: "auto" = حسب ذاكرة GPU، أو رقم 1/2/4/6/8
    SAVE_EVERY     = 250,    # حفظ نقطة تفتيش كل N خطوة (تُنسخ فوراً إلى Drive)
    TRAIN_WORKERS  = 2,      # عمال تحميل بيانات التدريب
    # النسخ إلى Google Drive (لحظي عند كل حفظ نقطة — بلا عملية مراقبة)
    DRIVE_BACKUP   = True,   # نسخ كل نقطة جديدة إلى Drive فور حفظها (من داخل التدريب)
    DRIVE_CKPT_KEEP = 2,     # كم نقطة تفتيش نُبقي على Drive (نقطة ≈ 0.4GB)
    # التخزين الدائم (HuggingFace Hub)
    HF_TOKEN       = "",     # رمز من hf.co/settings/tokens بصلاحية Write — للرفع الدائم (الخلية 16)
    HF_REPO_ID     = "",     # فارغ = <اسم-حسابك>/rafidain-seedvc-iraqi تلقائياً
    HF_PRIVATE     = True,   # مستودع خاص لا يراه أحد غيرك
    HF_ENDPOINT    = "",     # مرآة HuggingFace إن لزم (مثال: https://hf-mirror.com) — اتركها فارغة عادة
    # تنظيف
    CLEAN_AFTER     = False, # حذف بيانات التدريب المفكوكة بعد التصدير (يوفر مساحة — الخلية 17)
)

# ---------- 4) إعادة تحميل الإعدادات المحفوظة (استئناف بعد أي انقطاع) ----------
CFG = dict(DEFAULTS)
FORCE_RESET = False   # اجعلها True مؤقتاً لإعادة كل الإعدادات إلى الافتراضية
if CFG_PATH.exists() and not FORCE_RESET:
    try:
        saved = json.loads(CFG_PATH.read_text(encoding="utf-8"))
        CFG.update({k: saved[k] for k in DEFAULTS if k in saved})
        print(f"[إعدادات] أُعيد تحميلها من {CFG_PATH} (عدّل ثم أعد التشغيل، أو FORCE_RESET=True للبدء من جديد)")
    except Exception as e:
        print(f"[إعدادات] تعذر قراءة المحفوظة ({e}) — نستخدم الافتراضية")
# ← عدّل أي قيمة هنا ثم أعد تشغيل الخلية، مثال:
#    CFG["STEPS"] = 1000
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---------- 5) المسارات الثابتة ----------
SEEDVC   = ROOT / "seed-vc"                    # مستودع Seed-VC (المدرب الأصلي)
RAFIDAIN = ROOT / "rafidain"                   # مستودع رافِدَين (الخدمة + خط البيانات)
DATA_DIR = ROOT / "data" / "iraqi_finetune"    # بيانات التدريب المجهزة (Opus 24kHz) — من الأرشيف الجاهز
RUN_DIR  = SEEDVC / "runs" / CFG["RUN_NAME"]   # نقاط التدريب والسجلات (محلي سريع)
EXPORTS  = ROOT / "exports"                    # حزم النموذج النهائية (محلي)
READY_TGZ = ROOT / "rafidain_current_sources_ready.tar.gz"   # الأرشيف الجاهز (محلي)
READY_STAMP = ROOT / ".ready_archive_ok"       # بصمة «الأرشيف مفكوك وسليم»
# ملاحظة: لا بيئة بايثون منفصلة في هذا الدفتر — كل المنطق يعمل داخل نواة
# الجلسة نفسها بالاستيراد المباشر؛ الحزم تُثبَّت في النواة (الخلية 5).

# مراياها على Drive (تُنشأ عند الحاجة — لا الآن إن لم يكن موصولاً)
def drive_run_dir():
    return (DRIVE_ROOT / "runs" / CFG["RUN_NAME"]) if (DRIVE_OK and DRIVE_ROOT) else None
def drive_export_base():
    return (DRIVE_ROOT / "exports") if (DRIVE_OK and DRIVE_ROOT) else None
def drive_data_dir():
    # نسخة الأرشيف على Drive (اختيارية — توفر إعادة التنزيل عند انقطاع الجلسة)
    return (DRIVE_ROOT / "data") if (DRIVE_OK and DRIVE_ROOT) else None

for p in (ROOT, EXPORTS):
    p.mkdir(parents=True, exist_ok=True)

# ---------- 6) بيئة الأوامر ----------
# (متغيرات البيئة فقط هنا — أداة أوامر النظام sh() تُعرَّف في الخلية 4،
#  ومنطق المشروع كله يُستدعى بالاستيراد المباشر من الخلية 6 وما بعدها)
ENV = dict(os.environ)
ENV["PYTHONUNBUFFERED"] = "1"
ENV["HF_HUB_DISABLE_XET"] = "1"           # تنزيل HTTP قابل للاستئناف — Xet يقيّد IPs Colab (خيار الخط الرسمي)
ENV["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
if CFG["HF_ENDPOINT"]:
    ENV["HF_ENDPOINT"] = CFG["HF_ENDPOINT"]

# ---------- 7) ملخص ----------
print("=" * 64)
print(f"جذر العمل (محلي سريع) : {ROOT}")
print(f"التخزين الدائم (Drive) : {DRIVE_ROOT if DRIVE_OK else 'غير موصول — اعتمد على الخلية 16 لاحقاً'}")
print("=" * 64)
print(f"  • البيانات          : أرشيف جاهز {CFG['READY_ARCHIVE_SIZE']/1e9:.2f}GB (بلا إعادة معالجة)")
print(f"  • التدريب            : {CFG['STEPS']} خطوة، حفظ كل {CFG['SAVE_EVERY']}, دفعة {CFG['BATCH']}")
print(f"  • اسم التشغيل        : {CFG['RUN_NAME']}")
print(f"  • نسخ Drive          : {'لحظي عند كل حفظ (آخر %d نقاط)' % CFG['DRIVE_CKPT_KEEP'] if (DRIVE_OK and CFG['DRIVE_BACKUP']) else ('غير متاح (Drive غير موصول)' if not DRIVE_OK else 'معطّل')}")
print(f"  • رمز HuggingFace    : {'مضبوط (سيُرفع للأبد في الخلية 16)' if CFG['HF_TOKEN'] else 'غير مضبوط (اختياري — الرفع الدائم)'}")
print(f"\n[المسارات]  البيانات: {DATA_DIR}")
print(f"            التدريب : {RUN_DIR}")
print(f"            التصدير : {EXPORTS}")
print(f"\nملاحظة: الرمز السري (إن وضعته) يُحفظ في {CFG_PATH} — على Drive الخاص بك فقط.")

In [ ]:
# ============================================================ #
#  الخلية ٣ — فحص البيئة: GPU / قرص / أدوات                    #
#  (يحدد حجم الدفعة تلقائياً حسب ذاكرة GPU المكتشفة)           #
#  (nvidia-smi استعلام نظام للقراءة فقط — عبر sh())           #
# ============================================================ #
import os, shutil, re, json, sys
from pathlib import Path

def gpu_query():
    try:
        # استعلام nvidia-smi عبر مُشغّل أوامر النظام (قراءة معلومات العتاد فقط)
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            outf = Path(td) / "gpu.txt"
            sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader,nounits", "-f", str(outf)],
               check=False, echo=False)
            out = outf.read_text().strip().splitlines() if outf.exists() else []
        return [l.strip() for l in out if l.strip()]
    except Exception:
        return []

gpus = gpu_query()
if gpus:
    for i, g in enumerate(gpus):
        parts = [p.strip() for p in g.split(",")]
        name, vram_mib, driver = parts[0], float(parts[1]), parts[2]
        print(f"[GPU {i}] {name} — ذاكرة {vram_mib/1024:.1f}GB — مشغل {driver}")
    if any("T4" in g for g in gpus):
        print("        [T4] الجلسة المجانية — 3000 خطوة تستغرق نحو 2-4 ساعات. اصبر أو ترقَّ إلى L4.")
else:
    print("[GPU] لا يوجد GPU! من القائمة: Runtime <- Change runtime type <- GPU ثم أعد تشغيل")
    print("        الجلسة (Runtime <- Run all) — تجهيز البيانات يعمل بدون GPU، التدريب لا.")

def auto_batch(vram_gb):
    if vram_gb >= 70: return 8
    if vram_gb >= 40: return 6
    if vram_gb >= 20: return 4
    if vram_gb >= 15: return 2
    return 1

if gpus:
    vram_gb = float([p.strip() for p in gpus[0].split(",")][1]) / 1024
    CFG["_BATCH"] = auto_batch(vram_gb)
else:
    vram_gb = 0.0
    CFG["_BATCH"] = 2
if str(CFG["BATCH"]).lower() == "auto":
    BATCH = CFG["_BATCH"]
    print(f"[دفعة] تلقائي => BATCH = {BATCH} (حسب {vram_gb:.0f}GB ذاكرة)")
else:
    BATCH = int(CFG["BATCH"])
    print(f"[دفعة] يدوي  => BATCH = {BATCH}")
CFG["_BATCH"] = BATCH
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- المعالج والذاكرة ----
cpu = os.cpu_count() or 4
try:
    mem_kb = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text()).group(1))
    print(f"[نظام] {cpu} نواة معالج — ذاكرة RAM ≈ {mem_kb/1e6:.1f}GB — بايثون النواة {sys.version_info.major}.{sys.version_info.minor}")
except Exception:
    print(f"[نظام] {cpu} نواة معالج — بايثون النواة {sys.version_info.major}.{sys.version_info.minor}")

# ---- القرص المحلي (الأرشيف الجاهز أنقص الحاجة كثيراً) ----
du = shutil.disk_usage(ROOT)
need = 14  # GB: أرشيف مفكوك + بيئة + نقاط أساسية + نقاط تدريب
print(f"[قرص محلي] المتاح على {ROOT}: حر {du.free/1e9:.1f}GB من {du.total/1e9:.1f}GB")
if du.free / 1e9 < need + 2:
    print(f"          [تحذير] المسار يحتاج ~{need}GB — حرر مساحة أو شغّل خلية التنظيف (17)")
else:
    print(f"          [جيد] يكفي لإعدادك الحالي (~{need}GB مطلوبة تقريباً — الأرشيف الجاهز وفّر نصف حاجة النسخة السابقة)")

# ---- Drive ----
if DRIVE_OK and DRIVE_ROOT:
    try:
        ddu = shutil.disk_usage(DRIVE_ROOT)
        print(f"[Drive] الحصة: حر {ddu.free/1e9:.1f}GB من {ddu.total/1e9:.1f}GB — النسخ الاحتياطية تحتاج ~2GB + الأرشيف 4.9GB إن اخترت حفظه")
    except Exception:
        print("[Drive] موصول (تعذر قراءة الحصة)")

# ---- الأدوات ----
for tool in ("ffmpeg", "ffprobe", "git", "curl"):
    print(f"[أداة] {tool:<8}: {'متوفر' if shutil.which(tool) else 'غير متوفر — تثبّته الخلية 4'}")

print("\n[فحص مبدئي] أكمل إلى الخلية 4 (التبعيات).")

In [ ]:
# ============================================================ #
#  الخلية ٤ — أدوات النظام + مُشغّل أوامر النظام sh()          #
#  (apt فقط للأدوات: ffmpeg/git — وsh() تُستخدم لاحقاً لأوامر #
#   النظام الضرورية فقط: apt-get / git / tar / curl / pip)    #
#  Colab يعمل بصلاحيات root عادة — بلا sudo                   #
#  هذه الخلية آخر موضع لأوامر النظام: كل منطق المشروع بعد    #
#  الخلية 5 يعمل داخل النواة بالاستيراد المباشر — بلا أي     #
#  عملية ابن للتدريب أو الاستدلال أو خط البيانات             #
# ============================================================ #
import shutil, subprocess
from pathlib import Path

def have(x):
    return shutil.which(x) is not None

def sh(cmd, cwd=None, env=None, check=True, echo=True):
    # مُشغّل أوامر النظام: يشغّل أداة نظام (apt/git/tar/curl/pip) ويطبع
    # مخرجاتها لحظياً داخل الخلية ويعيد رمز الخروج.
    # ملاحظة سياسة الاستخدام: تُستخدم حصرياً لأدوات النظام أعلاه — أما التدريب
    # والاستدلال وخط البيانات فتعمل داخل نواة الدفتر بالاستيراد المباشر.
    cmd = [str(c) for c in cmd]
    if echo:
        print("$ " + " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None,
                            env=env or ENV, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"فشل الأمر (رمز {rc}): {' '.join(cmd)}")
    return rc

missing = [p for p in ("ffmpeg", "git") if not have(p)]
if missing:
    print(f"تثبيت حزم النظام: {', '.join(missing)} ...")
    sh(["apt-get", "update", "-qq"], check=False)
    sh(["apt-get", "install", "-y", "-qq", *missing])
for tool in ("ffmpeg", "git", "curl", "tar"):
    print(f"  {tool:<8}: {'متوفر' if have(tool) else 'غير متوفر!'}")
print("\n[جاهز] أدوات النظام كاملة ومُشغّل أوامر النظام معرّف — لا بيئة بايثون منفصلة في هذا الدفتر.")

In [ ]:
# ============================================================ #
#  الخلية ٥ — حزم بايثون داخل نواة الجلسة نفسها               #
#  لا بيئة منفصلة ولا عملية ابن لمنطق المشروع: التدريب       #
#  وخط البيانات ستعمل بالاستيراد داخل هذه النواة مباشرة،      #
#  لذا نثبّت الإصدارات المطلوبة هنا في نواة Colab ونضبطها     #
#  (نفس الإصدارات التي يثبتها الخط الرسمي seed_vc_iraqi_colab.sh)
# ============================================================ #
import importlib, sys
from importlib.metadata import version as pkg_version, PackageNotFoundError

def have_pkg(name):
    try:
        pkg_version(name)
        return True
    except PackageNotFoundError:
        return False

def pip_install(pkgs):
    # تثبيت الحزم في نواة الجلسة (العملية نفسها التي تدير الدفتر)
    rc = sh([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
    return rc

def pkg_ok(spec, import_name):
    """spec مثل 'torch==2.4.0' — يتحقق أن الإصدار المطلوب مستورد فعلاً في النواة."""
    want = spec.split("==")[1] if "==" in spec else None
    try:
        mod = importlib.import_module(import_name)
    except Exception:
        return False
    if want is None:
        return True
    return getattr(mod, "__version__", "?").startswith(want)

# ---- 1) PyTorch 2.4.0 cu121 — الإصدار الذي ينجح به الخط الرسمي (نحو 2.5GB) ----
if not pkg_ok("torch==2.4.0", "torch"):
    print("تثبيت PyTorch 2.4.0 + torchaudio (cu121) في نواة الجلسة — عدة دقائق ...")
    idx = ["--index-url", "https://download.pytorch.org/whl/cu121"]
    rc = pip_install(["torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0", *idx])
    if rc != 0:
        pip_install(["torch", "torchvision", "torchaudio", *idx])
    # بعد التثبيت: يجب إعادة تشغيل النواة لتفعيل الإصدار الجديد (torch محمّل مسبقاً)
    print("""
[مهم] ثُبّت PyTorch 2.4.0 — أعد تشغيل النواة الآن ثم أكمل الخلايا:
      Runtime <- Restart session (أو Restart and run all)
      (Colab يحمّل torch بإصدار قديم مسبقاً؛ إعادة التشغيل شرط لتفعيل 2.4.0 داخل النواة)
      الخلايا كلها استئنافية — لا شيء يُعاد تنزيله بعد إعادة التشغيل.""")
else:
    import torch
    print(f"PyTorch {torch.__version__} جاهز في النواة | CUDA: {torch.cuda.is_available()}")

# ---- 2) بقية الاعتماديات (نفس إصدارات الخط الرسمي، بالاستيراد داخل النواة) ----
need_imports = {
    "librosa": "librosa==0.10.2", "soundfile": "soundfile==0.12.1",
    "munch": "munch==4.0.0", "einops": "einops==0.8.0",
    "transformers": "transformers==4.46.3", "pyarrow": "pyarrow==17.0.0",
    "yaml": "pyyaml", "huggingface_hub": "huggingface_hub",
    "matplotlib": "matplotlib", "scipy": "scipy==1.13.1", "numpy": "numpy==1.26.4",
}
missing = []
for imp, spec in need_imports.items():
    try:
        importlib.import_module(imp)
    except Exception:
        missing.append(spec)
if missing:
    print(f"تثبيت الاعتماديات الناقصة داخل النواة: {len(missing)} حزمة ...")
    rc = pip_install(missing)
    if rc != 0:
        pip_install([s.split("==")[0] for s in missing])
else:
    print("الاعتماديات موجودة أصلاً في النواة — تخطّي التثبيت.")

# ---- 3) تحقق نهائي داخل النواة (استيراد فعلي — لا فحص في عملية خارجية) ----
import torch, torchaudio
assert torch.__version__.startswith("2.4."), (
    f"النواة تحمّل torch {torch.__version__} وليس 2.4.0 — أعد تشغيل النواة "
    "(Runtime <- Restart session) ثم Run all؛ الخلايا استئنافية")
print(f"\n[جاهز] النواة نفسها هي بيئة التشغيل: torch {torch.__version__} | "
      f"torchaudio {torchaudio.__version__} | CUDA {torch.cuda.is_available()}")
print("        التدريب والاستدلال سيعملان بالاستيراد المباشر داخل هذه النواة — بلا أي عملية ابن.")

In [ ]:
# ============================================================ #
#  الخلية ٦ — استنساخ المستودعات ثم استيراد وحداتها مباشرة     #
#  (كل منطق المشروع يُستخدم من الآن بالاستيراد — بلا سطر       #
#   أوامر لخط البيانات أو التدريب أو الاستدلال)               #
# ============================================================ #
import sys, importlib
from pathlib import Path

def git_clone(url, dst):
    if (dst / ".git").exists():
        print(f"تحديث {dst.name} ...")
        sh(["git", "-C", str(dst), "pull", "--ff-only"], check=False)
    else:
        print(f"استنساخ {dst.name} من {url} ...")
        sh(["git", "clone", "--depth", "1", url, str(dst)])

git_clone("https://github.com/Plachtaa/seed-vc.git", SEEDVC)
git_clone("https://github.com/Alqudimi/rafidain.git", RAFIDAIN)

# تحقق من الملفات الحرجة
for f in ("train.py", "inference.py", "configs/presets/config_dit_mel_seed_uvit_xlsr_tiny.yml"):
    assert (SEEDVC / f).exists(), f"ملف مفقود في seed-vc: {f}"
PIPE_SRC = RAFIDAIN / "scripts" / "data_research" / "iraqi_pipeline_v3.py"
assert PIPE_SRC.exists(), "iraqi_pipeline_v3.py مفقود في rafidain"
print(f"[جاهز] seed-vc   : {SEEDVC}")
print(f"[جاهز] rafidain  : {RAFIDAIN}")

# ---- استيراد خط بيانات العراق من المستودع المستنسخ (مصدر واحد للحقيقة) ----
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location("iraqi_pipeline", PIPE_SRC)
PIPELINE = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(PIPELINE)
print(f"[استيراد] iraqi_pipeline: {len(PIPELINE.CLIPS):,} مقطعاً / {PIPELINE.TOTAL_CLIP_HOURS:.0f} ساعة في الكتالوج")

# وحدة مزامنة النقاط من رافِدَين (restore/push_once — بلا خيوط خلفية)
sys.path.insert(0, str(RAFIDAIN))
from finetune import checkpoint_sync as CKPT_SYNC
print(f"[استيراد] finetune.checkpoint_sync: {CKPT_SYNC.__name__}")

# ---- مدرب واستدلال Seed-VC: استيراد مباشر داخل نواة الدفتر ----
# train.py وinference.py يفتحان configs/hifigan.yml و./checkpoints نسبياً إلى cwd،
# لذلك نجعل مجلد seed-vc هو دليل عمل النواة منذ الآن (وتبقى مساراتنا كلها مطلقة).
import os as _os
_os.chdir(SEEDVC)
sys.path.insert(0, str(SEEDVC))
import train as SEED_TRAIN          # المدرب الرسمي: SEED_TRAIN.Trainer(...)
import inference as SEED_INFER      # الاستدلال الرسمي: SEED_INFER.main(args-like)
print(f"[استيراد] seed-vc train.Trainer + inference.main — داخل نواة الدفتر مباشرة")

SEEDVC_CFG = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"
print(f"[مسار] إعداد التدريب الرسمي: {SEEDVC_CFG}")
print(f"[نواة] دليل العمل الحالي: {Path.cwd()} (seed-vc — لتصح المسارات النسبية للمدرب)")
print("\n[جاهز] المستودعات مستنسخة والوحدات مستوردة داخل النواة — تابع إلى خلية البيانات الجاهزة (7).")

<div dir="rtl">

## ٣) البيانات الجاهزة — أرشيف واحد بدل جلب المصادر ومعالجتها

النسخة الأولى من هذا الدفتر كانت تنزّل من خمسة مصادر (YODAS وغيرها) ثم تقطّع
وتُرمّز كل شيء من الصفر — ساعتان أو أكثر قبل أول خطوة تدريب. هنا بدل ذلك كله
**أرشيف واحد مُجهَّز مسبقاً**: `rafidain_current_sources_ready.tar.gz` (4.89GB) يحتوي
مجلد `iraqi_finetune/` وفيه المقاطع النهائية جاهزة (Opus 24kHz أحادي، 1–29.5 ثانية)
من كل المصادر العراقية — أي بالضبط ما ينتجه أمرا `small`+`yodas`+`prep` في خط
البيانات، لكن جاهزاً.

**كيف يتعامل الدفتر مع الأرشيف بلا تكرار؟**
1. إن وُجدت بصمة `.ready_archive_ok` (جلسة سابقة على نفس الجهاز) — تخطٍ كامل فوري.
2. وإلا إن وُجدت نسخة الأرشيف على Drive — تُنسخ منه مباشرة (أسرع وأضمن من الإنترنت).
3. وإلا يُنزَّل من الرابط المباشر بتنزيل **قابل للاستئناف** (curl -C -) — انقطاع
   التنزيل لا يعيد البداية.
4. قبل الفك يتحقق الدفتر من **الحجم الدقيق بالبايت** — نصف ملف مضغوط لا يُفك أبداً.
5. بعد الفك تُحذف نسخة tar.gz المحلية (المساحة تهم)، مع خيار نسخ الأرشيف إلى Drive
   لجلسة انقطاع قادمة أسرع (اختياري — مُفعّل افتراضياً إن توفرت المساحة).

> خلاصة: في أسوأ الحالات (جلسة أولى بلا Drive) تدفع ثمن التنزيل مرة واحدة؛ وفي كل
> الحالات الأخرى البيانات تظهر جاهزة خلال دقيقة إلى دقائق معدودة.

</div>

In [ ]:
# ============================================================ #
#  الخلية ٧ — البيانات الجاهزة: تنزيل الأرشيف وفكّه (مرة واحدة) #
#  rafidain_current_sources_ready.tar.gz (4.89GB)              #
#  بلا إعادة تنزيل من المصادر وبلا أي إعادة معالجة            #
# ============================================================ #
import os, shutil, time, json
from pathlib import Path

URL    = CFG["READY_ARCHIVE_URL"]
EXPECT = int(CFG["READY_ARCHIVE_SIZE"])          # الحجم الدقيق بالبايت (تحقق الاكتمال)
INNER  = CFG["READY_DIR_NAME"]                   # المجلد الجذري داخل الأرشيف
DST    = READY_TGZ                               # /content/iraqi_seedvc/rafidain_..._ready.tar.gz

def archive_complete(p: Path) -> bool:
    return p.exists() and p.stat().st_size == EXPECT

def extract_ready():
    """فك الأرشيف: محتوى iraqi_finetune/ يصبح بيانات التدريب في DATA_DIR."""
    t0 = time.time()
    print(f"فك الأرشيف إلى {DATA_DIR} ... (2-4 دقائق عادة)")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    # ننزع الضغط في مجلد مؤقت ثم ننقل محتوى iraqi_finetune — أمان من أي اختلاف هيكلي
    tmp = ROOT / "_ready_tmp"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    sh(["tar", "-xzf", str(DST), "-C", str(tmp)])
    inner_dir = tmp / INNER
    assert inner_dir.is_dir(), f"بنية الأرشيف غير متوقعة: {INNER}/ غير موجود"
    moved = 0
    for src in inner_dir.iterdir():
        dst = DATA_DIR / src.name
        if dst.exists():
            if dst.is_dir():
                shutil.rmtree(dst)
            else:
                dst.unlink()
        shutil.move(str(src), str(dst))
        moved += 1
    shutil.rmtree(tmp)
    n = sum(1 for _ in DATA_DIR.rglob("*.ogg"))
    hours = 0.0
    idx = DATA_DIR / "_prep_index.jsonl"
    if idx.exists():
        for line in idx.read_text().splitlines():
            try:
                hours += json.loads(line)["dur"] / 3600
            except Exception:
                pass
    READY_STAMP.write_text(json.dumps({
        "archive": DST.name, "size": EXPECT,
        "extracted_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "ogg_files": n, "indexed_hours": round(hours, 1),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"فُكّت {moved} مدخلاً | {n:,} ملف Opus | فهرس: {hours:.1f} ساعة | {time.time()-t0:.0f} ثانية")

# ---------- 0) منجز أصلاً؟ ----------
if READY_STAMP.exists() and any(DATA_DIR.rglob("*.ogg")):
    n = sum(1 for _ in DATA_DIR.rglob("*.ogg"))
    print(f"[تخطّي] البيانات الجاهزة موجودة أصلاً: {n:,} ملف Opus في {DATA_DIR}")
    print("        (لبناء البيانات من المصدر بدل الأرشيف: احذف " + str(READY_STAMP) + " واستخدم سكربتات المستودع)")
# ---------- 1) نسخة محلية كاملة من تنزيل سابق؟ ----------
elif archive_complete(DST):
    print(f"[عثرنا] أرشيف محلي كامل من محاولة سابقة ({DST.stat().st_size/1e9:.2f}GB) — فك مباشرة بلا تنزيل")
    extract_ready()
else:
    # ---------- 2) نسخة Drive؟ (توفر إعادة التنزيل بعد انقطاع الجلسة) ----------
    drv_arc = (drive_data_dir() / DST.name) if drive_data_dir() else None
    used_drive = False
    if drv_arc and archive_complete(drv_arc):
        print(f"[Drive] نسخة كاملة من الأرشيف موجودة على Drive — النسخ منها أسرع من التنزيل ...")
        t0 = time.time()
        shutil.copy2(drv_arc, DST)
        print(f"[Drive] نُسخ في {time.time()-t0:.0f} ثانية")
        used_drive = True
    # ---------- 3) التنزيل القابل للاستئناف ----------
    if not archive_complete(DST):
        if DST.exists():
            print(f"[استئناف] يوجد ملف جزئي ({DST.stat().st_size/1e9:.2f}GB من {EXPECT/1e9:.2f}GB) — يُستكمل من حيث توقف")
        free_gb = shutil.disk_usage(ROOT).free / 1e9
        assert free_gb > EXPECT/1e9 + 7, f"المساحة لا تكفي (تحتاج {(EXPECT/1e9+7):.1f}GB)"
        print(f"تنزيل الأرشيف الجاهز ({EXPECT/1e9:.2f}GB) — عادة 5-20 دقيقة حسب الشبكة ...")
        t0 = time.time()
        # curl -C - : استئناف تلقائي؛ المحاولات المتكررة تكسر الاتصالات المعلقة
        rc = sh(["bash", "-c",
                 f"curl -L -C - --retry 8 --retry-delay 3 --connect-timeout 30 "
                 f"-o '{DST}' '{URL}'"], check=False)
        if not archive_complete(DST):
            raise RuntimeError(
                f"التنزيل لم يكتمل (حجم الملف {DST.stat().st_size if DST.exists() else 0:,} من {EXPECT:,} بايت)\n"
                "أعد هذه الخلية — التنزيل قابل للاستئناف وسيتابع من حيث توقف.")
        print(f"[تم] اكتمل التنزيل في {(time.time()-t0)/60:.1f} دقيقة")
    extract_ready()

    # ---------- 4) نسخ الأرشيف إلى Drive (سرعة الجلسة القادمة) — بعد إتمام كل شيء ----------
    if (not used_drive) and drive_data_dir() and CFG.get("DRIVE_BACKUP", True):
        try:
            ddu = shutil.disk_usage(DRIVE_ROOT)
            if ddu.free > EXPECT/1e9 + 1.5:
                print(f"\n[Drive] حفظ نسخة الأرشيف لجلسات قادمة ({EXPECT/1e9:.2f}GB) ...")
                t0 = time.time()
                shutil.copy2(DST, drive_data_dir() / DST.name)
                print(f"[Drive] حُفظت في {time.time()-t0:.0f} ثانية — الجلسة القادمة ستستغني عن التنزيل كلياً")
            else:
                print(f"\n[Drive] لا مساحة كافية لنسخة الأرشيف ({ddu.free/1e9:.1f}GB حر) — تخطّي (لا يؤثر)")
        except Exception as e:
            print(f"[Drive] تعذر حفظ نسخة الأرشيف ({e}) — لا يؤثر على العمل")

# ---------- 5) حذف المضغوط المحلي (وفّر 4.9GB — البصمة تحمي من إعادة التنزيل) ----------
if DST.exists() and READY_STAMP.exists() and not any(ROOT.glob("_keep_archive")):
    DST.unlink()
    print(f"[مساحة] حُذف المضغوط المحلي — الحر الآن {shutil.disk_usage(ROOT).free/1e9:.1f}GB")

print(f"\n[جاهز] بيانات التدريب: {DATA_DIR}")
print("        تابع إلى خلية بياناتك الخاصة (8، اختيارية) أو إحصاءات البيانات (9).")

In [ ]:
# ============================================================ #
#  الخلية ٨ — (اختيارية) دمج تسجيلاتك الخاصة في التدريب        #
#  على Colab: الشريط الجانبي الأيسر <- أيقونة المجلد <- تنقّل   #
#  إلى /content/iraqi_seedvc/raw/custom <- زر الرفع (أعلى      #
#  اللوحة) ثم أعد تشغيل هذه الخلية                             #
#  ستُقطَّع تلقائياً إلى مقاطع 1-29.5 ثانية وتُرمَّز Opus 24kHz  #
#  (نفس منطق خط البيانات الرسمي — عبر الاستيراد لا سطر الأوامر)#
# ============================================================ #
import sys, os, json, time, types
from multiprocessing import Pool
from pathlib import Path

CUSTOM = ROOT / "raw" / "custom"
CUSTOM.mkdir(parents=True, exist_ok=True)
EXTS = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".webm"}
files = [p for p in sorted(CUSTOM.iterdir()) if p.is_file() and p.suffix.lower() in EXTS]
print(f"مجلد بياناتك: {CUSTOM}")
print(f"ملفات صوتية موجودة: {len(files)}")

if not files:
    print(R"""
لا توجد ملفات بعد — لتفعيل ذلك:
  1) من لوحة الملفات الجانبية في Colab: تنقّل إلى المجلد أعلاه ثم ارفع ملفاتك بزر الرفع
     (يدعم: wav / flac / mp3 / m4a / opus / ogg — الرفع الكبير يستغرق دقائق)
  2) الأفضل: كلام نظيف بلا موسيقى أو ضجيج، 1-30 ثانية أو أطول (تُقطَّع تلقائياً)
  3) أعد تشغيل هذه الخلية — سيلتقطها التدريب تلقائياً في الخلية 12
(تخطَّ هذه الخلية إن لم يكن لديك بيانات خاصة — الأرشيف الجاهز كافٍ تماماً)""")
else:
    # التقطيع عبر وحدة خط البيانات المستوردة (الخلية 6) — بلا سطر أوامر
    a = types.SimpleNamespace(out=str(DATA_DIR))
    jobs = []
    for f in files:
        jobs.extend(PIPELINE._prep_job_src(a, str(f), "custom"))
    print(f"سيُنتج عنها {len(jobs)} مقطعاً (يُتخطى الموجود منها)")
    t0, ok_n, bad_n = time.time(), 0, 0
    with Pool(processes=min(16, os.cpu_count() or 4)) as pool:
        for out, dur, okk in pool.imap_unordered(PIPELINE._run_prep_job, jobs, chunksize=16):
            if okk:
                ok_n += 1
            else:
                bad_n += 1
    print(f"تم: {ok_n} مقطعاً ناجحاً | {bad_n} فاشلاً في {time.time()-t0:.0f} ثانية")
    # تحديث فهرس بيانات التدريب (نفس منطق cmd_prep في الوحدة)
    idx = DATA_DIR / "_prep_index.jsonl"
    prev = {}
    if idx.exists():
        for line in idx.read_text().splitlines():
            try:
                r = json.loads(line)
                prev[r["file"]] = r["dur"]
            except Exception:
                pass
    for j in jobs:
        prev[os.path.relpath(j[3], DATA_DIR)] = j[2]
    prev = {f: d for f, d in prev.items() if (DATA_DIR / f).exists()}
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    idx.write_text("".join(json.dumps({"file": f, "dur": d}) + "\n" for f, d in sorted(prev.items())))
    print("فهرس بيانات التدريب محدّث — أكمل إلى الخلية 9.")

In [ ]:
# ============================================================ #
#  الخلية ٩ — إحصاءات بيانات التدريب (من فهرس الأرشيف الجاهز)  #
#  تقرير الساعات/الملفات لكل مصدر — بلا فحص صوتيات ثقيل        #
# ============================================================ #
import json
from collections import defaultdict
from pathlib import Path

idx = DATA_DIR / "_prep_index.jsonl"
assert idx.exists(), "لا يوجد فهرس بيانات — شغّل خلية الأرشيف الجاهز (7) أولاً"

per, tot_h, tot_n = defaultdict(lambda: [0, 0.0]), 0.0, 0
for line in idx.read_text().splitlines():
    try:
        r = json.loads(line)
    except Exception:
        continue
    if not (DATA_DIR / r["file"]).exists():
        continue
    src = r["file"].split("/")[0]
    per[src][0] += 1
    per[src][1] += r["dur"] / 3600
    tot_n += 1
    tot_h += r["dur"] / 3600

# مقاطع موجودة فعلياً بلا فهرس (مثل custom غير المفهرس أعلاه) — تعدّ فقط
on_disk = sum(1 for _ in DATA_DIR.rglob("*.ogg"))

print("=" * 52)
print(f"{'المصدر':<12} {'الملفات':>9} {'الساعات':>9}")
print("-" * 52)
for k in sorted(per):
    print(f"{k:<12} {per[k][0]:>9,} {per[k][1]:>9.2f}")
print("-" * 52)
print(f"{'الإجمالي':<12} {tot_n:>9,} {tot_h:>9.2f}")
print("=" * 52)
print(f"ملفات Opus على القرص: {on_disk:,} (يشمل ما أُضيف خارج الفهرس)")
assert on_disk >= 50, f"مقاطع قليلة جداً ({on_disk}) — أعد خلية الأرشيف (7)"

# حفظ الإحصاءات مع بيانات التدريب (تُستخدم في تقرير التصدير لاحقاً)
stats = {"total_files": tot_n, "total_hours": round(tot_h, 3),
         "per_source": {k: {"files": v[0], "hours": round(v[1], 3)} for k, v in per.items()}}
(DATA_DIR / "_stats.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n[جاهز] {tot_h:.1f} ساعة عراقية جاهزة للتدريب — تابع إلى خلية نقاط النموذج (10).")

In [ ]:
# ============================================================ #
#  الخلية ١٠ — تنزيل نقاط النموذج الأساسية مسبقاً (≈2.2GB)     #
#  DiT_uvit_tat_xlsr_ema + XLSR-300m + HiFT + CAMPPlus +       #
#  OpenVoice — كلها في مجلد checkpoints داخل seed-vc           #
#  (يمنع تعطل بدء التدريب لاحقاً بسبب التنزيل، وقابل للاستئناف)#
#  استدعاء مباشر لدالة cmd_prefetch من وحدة خط البيانات        #
#  المستوردة في الخلية 6 — داخل نواة الدفتر، بلا أي عملية خارجية #
# ============================================================ #
import types
from pathlib import Path

# cmd_prefetch يقرأ huggingface_hub من النواة (مثبتة في الخلية 5)
# cwd هو seed-vc منذ الخلية 6 — مجلد checkpoints النسبي يصح تلقائياً
rc = PIPELINE.cmd_prefetch(types.SimpleNamespace(repo=str(SEEDVC)))
assert not rc, "فشل prefetch — راجع السجل أعلاه"

ck = SEEDVC / "checkpoints"
total = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e9 if ck.exists() else 0
print(f"\nحجم مخزون النقاط: {total:.2f}GB في {ck}")
print("[جاهز] كل المطلوب للتدريب موجود — تابع إلى مرحلة التدريب (الخلية 11).")

<div dir="rtl">

## مرحلة التدريب — داخل نواة الدفتر، بلا عمليات خلفية وبلا subprocess

- **نقطة البداية دائماً** الأوزان المنشورة `DiT_uvit_tat_xlsr_ema.pth` من `Plachta/Seed-VC`
  (لا تدريب من الصفر أبداً). إن وُجدت نقطة سابقة في مجلد التشغيل يُستأنف منها تلقائياً
  (سلوك المدرب الرسمي: يبحث عن أحدث `DiT_epoch_*_step_*.pth` ويكمل منها، ويُبقي آخر
  نقطتين فقط).
- **كيف يعمل هنا؟** `import train` من مستودع Seed-VC المستنسخ (تمت في الخلية 6)، ثم
  ننشئ `Trainer` ونستدعي `.train()` **داخل نواة الدفتر نفسها** — لا بيئة منفصلة ولا عملية
  ابن ولا سطر أوامر إطلاقاً. ترى شريط تقدم tqdm وأسطر `epoch E, step S, loss: L`
  تظهر مباشرة أمامك. هذا هو النمط الذي تسمحه سياسة Colab المجانية: استخدام تفاعلي
  حقيقي للنواة.
- **لماذا لا عملية خلفية ولا subprocess؟** النسخة الأولى كانت تطلق التدريب كعملية
  مستقلة تعيش بعد انتهاء الخلية، مع مراقب نسخ دائم وحلقة انتظار لساعات — وهذا
  يُصنَّف «استخداماً غير مسموح» في الطبقة المجانية فتُقطع الجلسة فجأة
  (Runtime disconnected). النسخة الثانية أزالت الخلفية لكن أبقت تشغيل بايثون بعملية
  ابن؛ هذه النسخة تشغّل المدرب نفسه داخل النواة — صفر عمليات خارجية لمنطق المشروع.
- **النسخ إلى Drive رغم ذلك — كيف؟** نغلّف `torch.save` (التي يستدعيها المدرب عند
  كل نقطة حفظ) بغلاف ينسخ النقطة الجديدة إلى Drive **فور حفظها من داخل النواة
  نفسها** — بلا مراقب ولا خيط خلفي. أقصى خسارة عند أي انقطاع = الخطوات منذ
  آخر `SAVE_EVERY` (250 افتراضياً ≈ دقائق معدودة على T4).
- **مخرجات التدريب**: سطر خسارة كل 10 خطوات، نقطة كل 250 خطوة (تُنسخ فوراً)، وفي
  النهاية `ft_model.pth` (أوزان DiT المضبوطة — كل ما يلزم للنشر والاستدلال) ثم تُنسخ
  هي أيضاً إلى Drive مع السجل الكامل.
- **الإعدادات الموصى بها من دليل المستودع (FINETUNING.md)**: دفعة 2 لذاكرة 16GB (T4)
  و4 لـ 24GB (L4)؛ 200–500 خطوة لمتحدث واحد؛ **1500–4000 خطوة لتكيّف عام مع اللهجة**
  (3000 افتراضياً هنا)؛ معدل التعلم مثبّت في الإعداد الرسمي (1e-5) — لا ترفعه لنموذج
  25M باراميتر.
- **انقطعت الجلسة؟** افتح الدفتر ← **Run all**: تُثبَّت الحزم في النواة، والأرشيف
  الجاهز يُفك من نسخة Drive إن وُجدت، والخلية 11 تستعيد أحدث نقطة من Drive، ويستأنف التدريب من
  عندها تلقائياً.

مدة تقديرية: 3000 خطوة على **T4 في حدود 2–4 ساعات تقريباً** (حسب أطوال المقاطع)، وعلى
**L4 نحو نصف ذلك**. شريط التقدم يظهر بعد أول دقيقة فتعرف سرعتك الفعلية بدقة.

> **تنبيه مهم**: أبقِ هذا التبويب مفتوحاً أثناء خلية التدريب — إغلاق التبويب قد
> يُوقف الجلسة (سلوك Colab المعروف). النقاط محفوظة على Drive أياً حدث، لذا حتى لو
> انقطعت الجلسة فجأة: Run all وتكمل من آخر نقطة.

</div>

In [ ]:
# ============================================================ #
#  الخلية ١١ — استعادة نقاط التدريب من Drive (بعد أي انقطاع)   #
#  تعيد أحدث نقاط التفتيش + السجلات من نسخة Drive الاحتياطية   #
#  إلى القرص المحلي فيستأنف التدريب من حيث توقف                #
#  (خلية استباقية: لا تفعل شيئاً إن كان المحلي أحدث أو لا نسخة)#
#  عبر وحدة finetune.checkpoint_sync المستوردة — بلا سطر أوامر #
# ============================================================ #
import shutil, re
from pathlib import Path

drv = drive_run_dir()

# النقاط المحلية الحالية
loc_ckpts = sorted(RUN_DIR.glob("DiT_epoch_*_step_*.pth"), key=CKPT_SYNC._step_of)
if loc_ckpts or (RUN_DIR / "ft_model.pth").exists():
    last = f" (حتى الخطوة {CKPT_SYNC._step_of(loc_ckpts[-1])})" if loc_ckpts else ""
    print(f"[محلي] توجد نقاط محلية أصلاً{last} — لا حاجة للاستعادة.")
    print("        (لفرض الاستعادة من Drive احذف مجلد التشغيل المحلي ثم أعد هذه الخلية)")
elif drv and Path(drv).exists():
    # استعادة أحدث نقطة من Drive (نفس منطق السكربت الرسمي — عبر الوحدة المستوردة)
    restored = CKPT_SYNC.restore(RUN_DIR, Path(drv))
    if restored is None:
        print(f"[Drive] لا توجد نسخة احتياطية سابقة في {drv} — تشغيل جديد كلياً.")
    else:
        ck = sorted(RUN_DIR.glob("DiT_epoch_*_step_*.pth"), key=CKPT_SYNC._step_of)
        if ck:
            print(f"\n[تم] أُعيدت أحدث نقطة إلى الخطوة {CKPT_SYNC._step_of(ck[-1])} — التدريب سيكمل منها (الخلية 12).")
        elif (RUN_DIR / "ft_model.pth").exists():
            print("\n[تم] النموذج النهائي ft_model.pth مستعاد — يمكنك القفز إلى التحقق (الخلية 13).")
        else:
            print("\n[تم] استُعيدت السجلات فقط — سيبدأ التدريب من الصفر.")
else:
    print("[استعادة] لا نسخة Drive (غير موصول أو لا سجل سابق) — تشغيل جديد كلياً.")
    print("          (التدريب يبدأ من الأوزان المنشورة DiT_uvit_tat_xlsr_ema.pth — هذا طبيعي)")

In [ ]:
# ============================================================ #
#  الخلية ١٢ — التدريب: داخل نواة الدفتر نفسها                 #
#  SEED_TRAIN.Trainer (المستورد في الخلية 6) يُنشأ ويُشغَّل      #
#  هنا مباشرة — بلا عملية ابن ولا سطر أوامر إطلاقاً         #
#  كل نقطة حفظ تُنسخ فوراً إلى Drive من داخل النواة (غلاف      #
#  torch.save) — والسجل الكامل يُحفظ في train_console.log     #
# ============================================================ #
import os, sys, time, json, glob, re, shutil, contextlib, io
from pathlib import Path
import torch

FT = RUN_DIR / "ft_model.pth"

if FT.exists():
    print(f"النموذج النهائي موجود أصلاً: {FT}")
    print("لتدريب جديد كلياً: غيّر RUN_NAME في الخلية 2 (أو احذف ft_model.pth ثم أعد هذه الخلية).")
else:
    BATCH    = int(CFG["_BATCH"] or 2)
    MAXS     = int(CFG["STEPS"])
    WORKERS  = int(CFG["TRAIN_WORKERS"])
    SAVE_EVERY = int(CFG["SAVE_EVERY"])
    DRV_RUN  = drive_run_dir() if CFG.get("DRIVE_BACKUP", True) else None
    KEEP     = int(CFG.get("DRIVE_CKPT_KEEP", 2))

    RUN_DIR.mkdir(parents=True, exist_ok=True)
    _log_path = RUN_DIR / "train_console.log"

    # ---- مرآة Drive اللحظية: غلاف حول torch.save ينسخ النقاط فور حفظها ----
    # (يُركَّب قبل إنشاء Trainer حتى تغطي كل عمليات الحفظ؛ النسخ متزامن داخل
    #  عملية التدريب نفسها — بلا خيط خلفي ولا عملية إضافية)
    _real_save = torch.save
    def _save_with_drive_mirror(obj, f, *a, **kw):
        _real_save(obj, f, *a, **kw)
        if DRV_RUN and isinstance(f, (str, os.PathLike)):
            p = str(f)
            base = os.path.basename(p)
            if base.startswith("DiT_epoch_") and base.endswith(".pth"):
                try:
                    os.makedirs(DRV_RUN, exist_ok=True)
                    dst = os.path.join(DRV_RUN, base)
                    tmp = dst + ".part"
                    shutil.copy2(p, tmp)
                    os.replace(tmp, dst)
                    # تشذيب النقاط القديمة على Drive (أبقِ آخر KEEP)
                    cks = sorted(glob.glob(os.path.join(DRV_RUN, "DiT_epoch_*_step_*.pth")),
                                 key=lambda x: int(re.search("step_([0-9]+)", x).group(1)))
                    for old in (cks[:-KEEP] if KEEP > 0 else []):
                        try: os.remove(old)
                        except OSError: pass
                    print(f"[drive-mirror] {base} -> Drive", flush=True)
                except Exception as e:
                    print(f"[drive-mirror] فشل نسخ {base}: {e} (التدريب يتابع)", flush=True)
            elif base == "ft_model.pth":
                try:
                    os.makedirs(DRV_RUN, exist_ok=True)
                    shutil.copy2(p, os.path.join(DRV_RUN, base))
                    shutil.copy2(_log_path, os.path.join(DRV_RUN, "train_console.log"))
                    print("[drive-mirror] ft_model.pth + السجل -> Drive (اكتمل التدريب)", flush=True)
                except Exception as e:
                    print(f"[drive-mirror] فشل نسخ النموذج النهائي: {e}", flush=True)
    torch.save = _save_with_drive_mirror

    # ---- سجل مزدوج: الشاشة (خلية الدفتر) + train_console.log ----
    class _Tee(io.TextIOBase):
        def __init__(self, original, path):
            self.original = original
            self.file = open(path, "a", encoding="utf-8", buffering=1)
        def write(self, s):
            self.original.write(s)
            self.file.write(s)
            return len(s)
        def flush(self):
            self.original.flush()
            self.file.flush()

    print(f"[train] بدء الضبط الدقيق: {MAXS} خطوة، دفعة {BATCH}، حفظ كل {SAVE_EVERY}", flush=True)
    print(f"[train] التشغيل داخل نواة الدفتر مباشرة (import train.Trainer) — بلا أي عملية ابن", flush=True)
    if DRV_RUN:
        print(f"[train] مرآة Drive مفعّلة: كل نقطة تُنسخ فور حفظها إلى {DRV_RUN} (أبقِ آخر {KEEP})", flush=True)

    t0 = time.time()
    try:
        with contextlib.redirect_stdout(_Tee(sys.stdout, _log_path)):
            trainer = SEED_TRAIN.Trainer(
                config_path=str(SEEDVC_CFG),       # إعداد seed-vc الرسمي (cwd = seed-vc منذ الخلية 6)
                pretrained_ckpt_path=None,         # يجد آخر نقطة محلية تلقائياً أو ينزّل الأساسية المنشورة
                data_dir=str(DATA_DIR),
                run_name=CFG["RUN_NAME"],
                batch_size=BATCH,
                num_workers=WORKERS,
                steps=MAXS,
                save_interval=SAVE_EVERY,
                max_epochs=1000,
                device="cuda:0",
            )
            trainer.train()
    finally:
        torch.save = _real_save    # أعِد torch.save الأصلي مهما حدث

    mins = (time.time() - t0) / 60
    if FT.exists():
        print(f"\n[اكتمل] التدريب انتهى في {mins:.0f} دقيقة — النموذج النهائي: {FT}")
        if DRV_RUN:
            print(f"[Drive] النقاط والسجل محفوظة في: {DRV_RUN}")
    else:
        tail = ""
        if _log_path.exists():
            tail = "\n".join(_log_path.read_text(errors="replace").splitlines()[-25:])
        raise RuntimeError(
            f"توقف التدريب دون إنتاج النموذج النهائي بعد {mins:.0f} دقيقة.\n"
            f"آخر السجل:\n{tail}\n"
            "أعد هذه الخلية — إن كانت هناك نقطة محفوظة سيُستأنف التدريب منها تلقائياً.")

# منحنى الخسارة من السجل (إن وُجدت نقاط كافية)
STEP_RE = re.compile(r"epoch ([0-9]+), step ([0-9]+), loss: ([0-9.eE+-]+)")
logp = RUN_DIR / "train_console.log"
pts = []
if logp.exists():
    for ln in logp.read_text(errors="replace").splitlines():
        m = STEP_RE.search(ln)
        if m:
            pts.append((int(m.group(2)), float(m.group(3))))
if len(pts) >= 2:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    xs, ys = zip(*pts)
    fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
    ax.plot(xs, ys, lw=1.3, color="#d62728")
    ax.set_xlabel("step")
    ax.set_ylabel("EMA loss")
    ax.set_title(f"loss curve - {CFG['RUN_NAME']} ({len(pts)} points)")
    ax.grid(alpha=0.3)
    chart = RUN_DIR / "loss_curve.png"
    fig.savefig(chart, dpi=110)
    plt.close(fig)
    print(f"منحنى الخسارة: {chart} ({len(pts)} نقطة)")
    # نسخه إلى Drive إن وُجد
    drv = drive_run_dir()
    if drv:
        try:
            shutil.copy2(chart, drv / chart.name)
        except Exception:
            pass

print("\nالخطوة التالية: خلية التحقق (13) — استماع حقيقي بالنقطة الناتجة.")

<div dir="rtl">

## التحقق والتصدير

قبل الاحتفال بالنموذج نجري **استدلالاً حقيقياً** بالنقطة الناتجة: مقطع مصدر من أمثلة Seed-VC يُحوَّل
بصوت مرجعي عراقي من بياناتك المجهزة، وتستمع للناتج داخل الدفتر مباشرة. بعدها نبني **حزمة التصدير**
وننسخها فوراً إلى Drive:

```
exports/iraqi_v1/                          (محلي + MyDrive/rafidain_seedvc/exports/iraqi_v1/)
├── seed_uvit_tat_xlsr_tiny_iraqi.pth      الأوزان المضبوطة (للنشر المباشر)
├── config_dit_mel_seed_uvit_xlsr_tiny.yml
├── training_report.json                   تقرير التدريب (البيانات والكتالوج والتاريخ)
├── _stats.json                            ساعات/ملفات كل مصدر
├── train_console.log                      سجل التدريب كاملاً
├── loss_curve.png                         منحنى الخسارة
└── README.md                              أوامر الاستخدام الجاهزة
```

</div>

In [ ]:
# ============================================================ #
#  الخلية ١٣ — التحقق: استدلال حقيقي بالنقطة المضبوطة          #
#  (مصدر من أمثلة Seed-VC + مرجع عراقي من بياناتك -> تشغيل)    #
#  استدعاء مباشر لـ SEED_INFER.main (المستوردة في الخلية 6)    #
#  داخل نواة الدفتر — بلا عملية خارجية وبلا سطر أوامر      #
# ============================================================ #
from IPython.display import Audio, display
from pathlib import Path
import types

FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — شغّل خلية التدريب (12) حتى الاكتمال"

# اختيار مرجع عراقي من البيانات الجاهزة (مثل منطق cmd_verify في خط البيانات)
ref = None
for d in ("yodas", "youssef", "lahgtna", "kharrufa", "shiekh", "safaa"):
    dd = DATA_DIR / d
    if dd.is_dir():
        cand = sorted(dd.glob("*.ogg"))[:5]
        if cand:
            ref = cand[0]
            break
assert ref is not None, "لا توجد مقاطع عراقية للمرجع — شغّل خلية الأرشيف (7)"
src = SEEDVC / "examples" / "source" / "source_s1.wav"
if not src.exists():
    src = next(iter(sorted((SEEDVC / "examples" / "source").iterdir())))
outdir = RUN_DIR / "verify_out"
outdir.mkdir(parents=True, exist_ok=True)

print(f"المصدر (إنجليزي): {src.name}")
print(f"المرجع (عراقي)  : {ref.relative_to(DATA_DIR)}")
print("تشغيل الاستدلال داخل النواة ... (1-3 دقائق)")

# الأرجس نفسها التي يمررها سطر الأمد لـ inference.py — لكن بلا سطر أوامر:
# نستدعي main() مباشرة على كائن args (SEED_INFER مستوردة من الخلية 6)
vc_args = types.SimpleNamespace(
    source=str(src),
    target=str(ref),
    output=str(outdir),
    diffusion_steps=16, length_adjust=1.0, inference_cfg_rate=0.7,
    f0_condition=False, auto_f0_adjust=False, semi_tone_shift=0,
    checkpoint=str(FT),
    config=str(SEEDVC_CFG),
    fp16=True,
)
try:
    SEED_INFER.main(vc_args)     # الاستدلال كاملاً داخل نواة الدفتر
    verify_ok = True
except Exception as e:
    print(f"[تحقق] فشل الاستدلال: {e}")
    verify_ok = False

outs = sorted(outdir.glob("vc_*.wav"))
if outs:
    print(f"\nناتج التحويل: {outs[0].name}  ({outs[0].stat().st_size/1e6:.1f} MB)")
    print("استمع للناتج (المصدر إنجليزي يحمل الطابع الصوتي العراقي المرجعي):")
    display(Audio(filename=str(outs[0])))
    for extra in outs[1:3]:
        print(f"ناتج إضافي: {extra.name}")
        display(Audio(filename=str(extra)))
else:
    print("لم يُنتج ملف صوتي — راجع السجل أعلاه. (غير قاتل: الأوزان ستصدَّر في الخلية 14 على أي حال)")

In [ ]:
# ============================================================ #
#  الخلية ١٤ — تصدير الحزمة النهائية                           #
#  أوزان + إعداد + تقرير + سجلات — محلياً ثم نسخها إلى Drive   #
#  استدعاء مباشر لـ PIPELINE.cmd_export داخل نواة الدفتر       #
#  (نفس الواجهة، بلا عملية خارجية وبلا سطر أوامر)            #
# ============================================================ #
import shutil, types, json
from pathlib import Path

FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — أكمل التدريب أولاً (الخلية 12)"
EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]

# استدعاء cmd_export من وحدة خط البيانات المستوردة (الخلية 6) — داخل النواة
PIPELINE.cmd_export(types.SimpleNamespace(
    run_dir=str(RUN_DIR),
    config=str(SEEDVC_CFG),
    data_dir=str(DATA_DIR),
    out=str(EXPORT_DIR),
))
assert (EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth").exists(), "فشل التصدير — راجع السجل"

# توثيق إضافي يُنسخ مع الحزمة (قابلية إعادة الإنتاج جزء من الجودة)
for extra in (RUN_DIR / "train_console.log", RUN_DIR / "loss_curve.png",
              RUN_DIR / "config_dit_mel_seed_uvit_xlsr_tiny.yml", DATA_DIR / "_stats.json"):
    if extra.exists():
        shutil.copy2(extra, EXPORT_DIR / extra.name)
        print(f"  + {extra.name}")

total = 0
print(f"\\nحزمة التصدير (محلي): {EXPORT_DIR}")
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        s = p.stat().st_size
        total += s
        print(f"  {p.name:<46} {s/1e6:9.2f} MB")
print(f"  {'الإجمالي':<46} {total/1e6:9.2f} MB")

# ---- النسخ الفوري إلى Drive (ديمومة فورية) ----
DRV_EXP = drive_export_base()
if DRV_EXP:
    print(f"\\nنسخ الحزمة إلى Drive: {DRV_EXP / CFG['RUN_NAME']} ...")
    (DRV_EXP / CFG["RUN_NAME"]).mkdir(parents=True, exist_ok=True)
    for p in sorted(EXPORT_DIR.rglob("*")):
        if p.is_file():
            shutil.copy2(p, DRV_EXP / CFG["RUN_NAME"] / p.name)
    print("[Drive] الحزمة محفوظة بشكل دائم — تنجو من أي انقطاع جلسة.")

In [ ]:
# ============================================================ #
#  الخلية ١٥ — أرشيف tar.gz واحد (في Drive + محلي)             #
# ============================================================ #
import shutil
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
assert (EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth").exists(), "شغّل خلية التصدير (14) أولاً"

base = EXPORTS / f"rafidain_seedvc_{CFG['RUN_NAME']}"
archive = shutil.make_archive(str(base), "gztar", root_dir=str(EXPORTS), base_dir=EXPORT_DIR.name)
size_mb = Path(archive).stat().st_size / 1e6
print(f"الأرشيف جاهز: {archive}  ({size_mb:.1f} MB)")

# ---- نسخه إلى Drive ----
if globals().get("DRIVE_OK") and globals().get("DRIVE_ROOT"):
    drv_arc = globals()["DRIVE_ROOT"] / "exports" / Path(archive).name
    shutil.copy2(archive, drv_arc)
    print(f"[Drive] نسخة دائمة: {drv_arc}")

print(R"""
لتنزيله إلى جهازك:
  • من Drive (الأسهل): drive.google.com <- مجلد rafidain_seedvc/exports <- نزّل الملف
  • أو من لوحة ملفات Colab (الشريط الجانبي): مجلد exports <- كليك يمين <- Download
  • أو أضف في خلية جديدة:  from google.colab import files; files.download("<مسار الأرشيف>")
""")


<div dir="rtl">

## الحفظ الدائم — ثلاث طبقات (حتى تنزّلها بعد سنة)

| الطبقة | أين تبقى | كم تدوم | كيف تنزّلها لاحقاً |
|---|---|---|---|
| 1. **Google Drive** | `MyDrive/rafidain_seedvc/` — نقاط أثناء التدريب (من مرآة الخلية 12) + حزمة التصدير + الأرشيف + نسخة أرشيف البيانات الجاهز | دائمة في حسابك مهما انقطعت جلسات Colab | من أي متصفح/جهاز عبر drive.google.com |
| 2. **HuggingFace Hub** | مستودع في حسابك (خاص افتراضياً) | **دائمة مجاناً — بلا علاقة بـ Google إطلاقاً** | `hf download <repo>` أو زر التنزيل من صفحة المستودع |
| 3. القرص المحلي `/content` | البيانات الجاهزة + البيئة + النقاط | تضيع عند انقطاع/إعادة تعيين الجلسة | (قابلة لإعادة البناء بالخلايا 5-7 — كلها استئنافية) |

> **الوصية**: الطبقة 2 هي ضمانتك الحقيقية للمدى البعيد — حتى لو فقدت حساب Google أو حُذف Drive،
> يبقى النموذج في HuggingFace قابلاً للتنزيل من أي مكان في العالم. تحتاج فقط رمز API بصلاحية كتابة
> من `hf.co/settings/tokens` تضعه في الخلية 2 ثم تشغّل الخلية 16.

ما يصل إلى Drive **أثناء التدريب** (مرآة الخلية 12، لحظياً عند كل حفظ): آخر نقطتين
`DiT_epoch_*.pth` + `ft_model.pth` عند الاكتمال + `train_console.log` + `loss_curve.png` —
كل ذلك في `MyDrive/rafidain_seedvc/runs/<اسم-التشغيل>/`، بمساحة لا تتجاوز ~2GB.

</div>

In [ ]:
# ============================================================ #
#  الخلية ١٦ — الرفع الدائم إلى HuggingFace Hub                #
#  النموذج يُخزَّن في حسابك أنت، مجاناً، بلا علاقة بـ Google     #
#  (أعد تشغيلها في أي وقت مستقبلاً — ولو بعد شهور)              #
#  عبر huggingface_hub مباشرة — بلا سكربت مؤقت ولا سطر أوامر   #
# ============================================================ #
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]

if not CFG["HF_TOKEN"]:
    print("لا يوجد رمز HuggingFace بعد — لا مشكلة: كل أصولك محفوظة على Drive،")
    print("ويمكنك تنفيذ الرفع الدائم في أي وقت لاحق (الخلية تعمل مستقلة):\\n")
    print("  1) أنشئ حساباً مجانياً في huggingface.co إن لم يكن لديك")
    print("  2) من hf.co/settings/tokens أنشئ New Token بصلاحية Write وانسخه")
    print('  3) في الخلية 2 اضبط:  CFG["HF_TOKEN"] = "hf_ضع-الرمز-هنا"  ثم أعد تشغيلها')
    print("  4) أعد تشغيل هذه الخلية — سترفع الحزمة كاملة إلى مستودع خاص في حسابك")
else:
    assert (EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth").exists(), "شغّل خلية التصدير (14) أولاً"
    # الرفع مباشرة داخل النواة — huggingface_hub مثبتة في الخلية 5 (بلا عملية خارجية)
    import time
    from huggingface_hub import HfApi
    api = HfApi(token=str(CFG["HF_TOKEN"]))
    user = api.whoami()["name"]
    repo_id = str(CFG.get("HF_REPO_ID", "")).strip() or f"{user}/rafidain-seedvc-iraqi"
    private = bool(CFG["HF_PRIVATE"])
    api.create_repo(repo_id, private=private, exist_ok=True)
    print(f"[hf] رفع {EXPORT_DIR} -> {repo_id} (خاص = {private}) ...")
    api.upload_folder(
        folder_path=str(EXPORT_DIR),
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"Rafidain Iraqi fine-tune ({time.strftime('%Y-%m-%d %H:%M')})",
    )
    print(f"[hf] REPO_URL: https://huggingface.co/{repo_id}")
    print(R"""
[تم] النموذج الآن في حسابك على HuggingFace — بشكل دائم.
لتنزيله في أي وقت مستقبلاً من أي جهاز (بلا Google ولا Colab):
    pip install -U "huggingface_hub[cli]"
    hf download <اسم-حسابك>/rafidain-seedvc-iraqi --local-dir ./mymodel
أو من بايثون:
    from huggingface_hub import snapshot_download
    snapshot_download("<اسم-حسابك>/rafidain-seedvc-iraqi", local_dir="./mymodel")
أو مباشرة من صفحة المستودع في المتصفح.""")


In [ ]:
# ============================================================ #
#  الخلية ١٧ — (اختيارية) تحرير مساحة القرص المحلي            #
#  تحذف البيانات القابلة لإعادة الجلب فقط — وتبقى النقاط       #
#  والحزم وكل ما على Drive كما هو                             #
#  (بيانات الأرشيف الجاهز تُعاد في دقائق من نسخة Drive)         #
# ============================================================ #
import shutil
from pathlib import Path

def dir_gb(p: Path) -> float:
    if not p.exists():
        return 0.0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

targets = [DATA_DIR, ROOT / "raw", SEEDVC / "checkpoints" / "hf_cache"]
labels = ["بيانات التدريب الجاهزة (الأرشيف)", "مصادر خام إضافية", "مخبأ تنزيل HF"]
print(f"المساحة الحرة الآن: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
for t, lb in zip(targets, labels):
    s = dir_gb(t)
    print(f"  {t.name:<14} {s:6.2f} GB   ({lb})")

if CFG["CLEAN_AFTER"]:
    for t in targets[:2]:   # البيانات والخام — أما مخبأ HF فيُحذف فقط لو الطلب صريح
        if t.exists():
            shutil.rmtree(t)
            print(f"  حُذف: {t}")
    if (ROOT / ".ready_archive_ok").exists():
        (ROOT / ".ready_archive_ok").unlink()
    import sys as _sys
    sh([_sys.executable, "-m", "pip", "cache", "purge"], check=False)
    print(f"المساحة الحرة بعد التنظيف: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
    print("كل ما يهم محفوظ على Drive (النقاط + الحزمة + الأرشيف) — لا خسارة دائمة.")
else:
    print(R"""
للتفعيل: اجعل  CFG["CLEAN_AFTER"] = True  في الخلية 2 ثم أعد تشغيل هذه الخلية.""")

In [ ]:
# ============================================================ #
#  الخلية ١٨ — النشر: كيف تستخدم النموذج المضبوط؟              #
#  (خلية إرشادية تطبع الأوامر الجاهزة بمساراتك الفعلية)        #
# ============================================================ #
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
W = EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth"
C = EXPORT_DIR / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

if not W.exists():
    print("شغّل خلية التصدير (14) أولاً حتى تتوفر الحزمة.")
else:
    print(f"""
==========================================================
 نشر النموذج المضبوط — ثلاثة مسارات جاهزة
==========================================================

(أ) استدلال فوري في هذه الجلسة — استيراد مباشر داخل النواة
    (الوحدات محمّلة أصلاً منذ الخلية 6 — بلا أي عملية خارجية):
    في خلية جديدة الصق:

      import types
      vc = types.SimpleNamespace(
          source=r"<صوت-المتكلم-الأصلي.wav>",
          target=r"<صوت-مرجعي-عراقي.wav>",   # مقطع عراقي من بياناتك
          output=r"./out",
          diffusion_steps=30, length_adjust=1.0, inference_cfg_rate=0.7,
          f0_condition=False, auto_f0_adjust=False, semi_tone_shift=0,
          checkpoint=r"{W}", config=r"{C}", fp16=True)
      SEED_INFER.main(vc)     # الاستدلال داخل النواة مباشرة

(ب) خدمة رافِدَين الكاملة على نفس جلسة Colab — REST + WebSocket + واجهة ويب
    بنفق عام مجاني (trycloudflare) للاستنساخ اللحظي من أي مكان.
    المستودع منسوخ هنا: {RAFIDAIN}
    في خلية جديدة الصق ما يلي (توجيه الخدمة إلى أوزانك المضبوطة):

      import os
      os.environ["RAFIDAIN__ENGINE__LOCAL_CHECKPOINT"] = r"{W}"
      os.environ["RAFIDAIN__ENGINE__LOCAL_CONFIG"]     = r"{C}"
      os.environ["RAFIDAIN__ENGINE__DEVICE"]           = "cuda"
      os.environ["RAFIDAIN__API_KEYS"]                 = r'["<مفتاح-API-عشوائي-خاص-بك>"]'
      # تشغيل سكربت تثبيت خدمة رافِدَين الرسمي (أمر نظام لتشغيل الخدمة):
      !bash {RAFIDAIN}/scripts/install/setup_colab.sh

    بعد دقائق يطبع رابطاً عاماً مؤقتاً:
      لوحة التحكم:  <الرابط>/web   |   البث الحي:  <الرابط>/live   |   API:  <الرابط>/docs
    (متغيرا البيئة LOCAL_CHECKPOINT / LOCAL_CONFIG هما المساران أعلاه — يُقرآن مباشرة من البيئة)

(ج) على أي جهاز GPU آخر (خادمك / RunPod / غيره):
    1) نزّل الحزمة من Drive (rafidain_seedvc/exports) أو من HuggingFace (الخلية 16)
    2) git clone https://github.com/Plachtaa/seed-vc && cd seed-vc
    3) python inference.py --checkpoint <الحزمة>/seed_uvit_tat_xlsr_tiny_iraqi.pth \\
           --config <الحزمة>/config_dit_mel_seed_uvit_xlsr_tiny.yml \\
           --source a.wav --target ref.wav --output ./out
    أو خدمة رافِدَين الكاملة بنفس متغيرَي البيئة في (ب) — الدليل: DEPLOYMENT.md و WEBUI.md

تذكير أخلاقي: استنساخ صوت شخص حقيقي يتطلب موافقته الصريحة الموثقة،
وأفصح للطرف الآخر عند ما يوجبه القانون — راجع قسم الأمان في README المستودع.
""")

<div dir="rtl">

## حل المشكلات الشائعة

| المشكلة | السبب | الحل |
|---|---|---|
| `Runtime disconnected` قديماً | النسخة الأولى كانت تشغّل التدريب كعملية خلفية + مراقب دائم + حلقة انتظار — أنماط «استخدام غير مسموح» في الطبقة المجانية | **أُزيلت كلها في هذه النسخة**: التدريب داخل نواة الخلية 12 مباشرة — لا عمليات خفية ولا عملية ابن إطلاقاً |
| انقطعت الجلسة أثناء التدريب | حد طبيعي لجلسات Colab | `Run all` — البيئة تُبنى، الأرشيف يُفك من Drive، والخلية 11 تستعيد آخر نقطة، والتدريب يكمل تلقائياً |
| انقطع تنزيل الأرشيف في المنتصف | شبكة | أعد الخلية 7 — `curl -C -` يستكمل من حيث توقف (تحقق الحجم الدقيق بالبايت قبل أي فك) |
| نفدت مساحة القرص | ت accumulation البيانات | خلية التنظيف (17) — أو قلّل مساحة بنسخ أرشيف Drive فقط عند الحاجة |
| `CUDA out of memory` أثناء التدريب | دفعة أكبر من الذاكرة | الخلية 2: `CFG["BATCH"] = 1` ثم أعد الخليتين 3 و12 |
| فشل تنزيل نقطة HF | شبكة/قيود IP | أعد الخلية 10 — التنزيل قابل للاستئناف؛ أو اضبط `HF_ENDPOINT` مرآة في الخلية 2 |
| التدريب بطيء جداً | T4 + مقاطع طويلة | طبيعي على الطبقة المجانية؛ أو قلّل `STEPS`؛ L4 يضاعف السرعة |
| فشل `verify` | مقطع مرجعي شاذ | غير قاتل — الأوزان تُصدَّر على أي حال (الخلية 14) |

**بعد أي انقطاع**: افتح الدفتر ← `Run all` ← كل شيء يستأنف: الحزم (دقائق)،
البيانات (من Drive: دقائق / من الإنترنت: حتى 20 دقيقة)، النقاط (ثوانٍ)، والتدريب
يكمل من آخر خطوة محفوظة. لا يوجد سيناريو يبدأ فيه الدفتر من الصفر إلا مرة واحدة.

</div>

<div dir="rtl">

## الخلاصة — ماذا حصلت عليه؟

1. **نموذج مضبوط** `seed_uvit_tat_xlsr_tiny_iraqi.pth` — Seed-VC tiny (25M باراميتر)
   تكيّف على الساعات العراقية الجاهزة، عبر تدريب داخل نواة الدفتر مباشرة (import train ← Trainer) بلا أي عملية خلفية أو عملية ابن.
2. **ثلاث نسخ دائمة**: Drive (نقاط + حزمة + أرشيف)، tar.gz واحد قابل للتنزيل،
   وHuggingFace Hub (اختياري لكنه الأضمن).
3. **قابلية استئناف كاملة**: كل خلية تتخطى ما أنجز، والأرشيف الجاهز جعل مرحلة
   البيانات دقائق بدل ساعات، والتدريب يستأنف من آخر نقطة.
4. **مسارات نشر جاهزة**: استدلال فوري (استيراد inference)، خدمة رافِدَين الكاملة
   (REST/WebSocket/ويب) على نفس الجلسة، أو أي جهاز GPU آخر.

**الخطوة التالية المقترحة**: شغّل التحقق (13) واستمع للناتج — ثم إن أعجبك، ارفع
النقطة إلى HuggingFace (16) وجرّب خدمة البث الحي (18-ب) بصوتك أنت.

</div>